in the previous notebook i implemented a transformer block in a torch module, but pytorch serves a pre-cooked transformer blocks, multi-head attentions, etc. with all the nook and crannies so i use that in this notebook. furthermore, i implement with direction from the paper 'TinyStories: How Small Can Language Models Be and Still Speak Coherent English?' by Eldan and Li (2023), which we will hereafter refer to as 'the paper'. they describe that even very small transformer models are able to get sub 1.5 loss (our best LSTM model was a 2.9 loss).

this would suggest a perplexity of 4.48 which would be excellent. first, we change our tokenizer to use the GPT-Neo tokenizer, and thus re-tokenise our training set.

In [4]:
import pandas as pd
import numpy as np
import os
from transformers import AutoTokenizer

In [2]:
import sys
from pathlib import Path
cwd = Path.cwd()
ROOT = next(p for p in (cwd, *cwd.parents) if (p / "src").is_dir())

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

we first pull in our raw training samples.

In [16]:
data_path = 'C:/data/tinystories/data'
data_files = os.listdir(data_path)
train_files = [d for d in data_files if 'train' in d]
test_files = [d for d in data_files if 'validation' in d]

# pull in train data and isolate text as a list
train_data = pd.concat([pd.read_parquet(f'{data_path}/{d}') for d in train_files])
train_text = train_data['text'].to_list()

test_data = pd.concat([pd.read_parquet(f'{data_path}/{d}') for d in test_files])
test_text = test_data['text'].to_list()

then, declare the gpt-neo tokeniser. we can then define an iterator for our data set, and retrain tokens

In [ ]:
tokeniser = AutoTokenizer.from_pretrained('EleutherAI/gpt-neo-125M', use_fast=True)

def batch_iterator(dataset, batch_size=1_000):
    for i in range(0, len(dataset), batch_size):
        yield dataset[i:i + batch_size]

trained_tokeniser = tokeniser.train_new_from_iterator(batch_iterator(train_text), vocab_size=9_999)

now, save down the tokeniser, and re-tokenise our training and validation set.

In [ ]:
def write_token_file(dataframe, output_path, tokeniser, batch_size=512):
    total_tokens = 0
    total_documents = 0

    with open(output_path, "wb") as f:
        for start in range(0, len(dataframe), batch_size):
            texts = dataframe["text"].iloc[start:start + batch_size].tolist()
            # encodes the text using the tokeniser
            encoded = tokeniser(texts, add_special_tokens=False, padding=False, truncation=False,)

            for token_ids in encoded["input_ids"]:
                # add end-of-sequence token
                token_ids.append(tokeniser.eos_token_id)

                # save down as numpy array, these can be streamed later on
                arr = np.asarray(token_ids, dtype=np.uint16)
                arr.tofile(f)

                total_tokens += len(arr)
                total_documents += 1

            if start % (batch_size * 100) == 0:
                progress = f"{start:,}/{len(dataframe):,} documents"
                print(f"{progress:<35} :: "f"{total_tokens:,} tokens")

            del texts
            del encoded

    print(f"Saved {total_documents:,} documents / " f"{total_tokens:,} tokens -> {output_path}")

create a `numpy` array binary file for testing and training files.

In [21]:
trained_tokeniser.save_pretrained('C:/dev/storyteller/weights/gpt-neo-10k-tokeniser')

write_token_file(train_data, 'C:/data/tinystories/processed/train_gpt_neo.bin', trained_tokeniser)
write_token_file(test_data, 'C:/data/tinystories/processed/test_gpt_neo.bin', trained_tokeniser)

0/2,119,719 documents               :: 98,481 tokens
51,200/2,119,719 documents          :: 11,214,811 tokens
102,400/2,119,719 documents         :: 22,103,446 tokens
153,600/2,119,719 documents         :: 33,636,785 tokens
204,800/2,119,719 documents         :: 45,358,622 tokens
256,000/2,119,719 documents         :: 56,783,922 tokens
307,200/2,119,719 documents         :: 68,125,888 tokens
358,400/2,119,719 documents         :: 79,323,183 tokens
409,600/2,119,719 documents         :: 90,416,588 tokens
460,800/2,119,719 documents         :: 101,566,232 tokens
512,000/2,119,719 documents         :: 112,866,636 tokens
563,200/2,119,719 documents         :: 124,199,086 tokens
614,400/2,119,719 documents         :: 135,378,087 tokens
665,600/2,119,719 documents         :: 146,718,310 tokens
716,800/2,119,719 documents         :: 158,391,889 tokens
768,000/2,119,719 documents         :: 169,918,255 tokens
819,200/2,119,719 documents         :: 180,912,882 tokens
870,400/2,119,719 documents

In [15]:
tokeniser = AutoTokenizer.from_pretrained('C:/dev/storyteller/weights/gpt-neo-10k-tokeniser')

there is now everything required to train the model. the paper details the model architecture as a 'GPT-Neo' model, which is a model trained by EleutherAI, and is supposed to replicate the earlier GPT-2/GPT-3 models. looking at the `33M` parameter model, the model setup is the below.

- absolute positional embeddings
- hidden layer size of 768 neurons
- feed forward size of 3072 neurons (within transformer block)
- 16 head multi-head transformer
- 4 blocks
- context window of 512 tokens

a *transformer block* first mixes information between tokens using attention, then transforms each token's features with a feed-forward network. our block normalises before each step and adds its input back through a residual connection:

$$
\begin{aligned}
    U &= X + \operatorname{Attention}(\operatorname{LayerNorm}_1(X)), \\
    Y &= U + \operatorname{FeedForward}(\operatorname{LayerNorm}_2(U)).
\end{aligned}
$$

*multi-head attention* runs $h$ attention heads in parallel. each head learns its own query, key and value projections from the full input, producing $Q_r, K_r, V_r \in \mathbb{R}^{T \times d_h}$, where $T$ is the number of tokens and $d_h=d/h$. with $d=768$ and $h=16$, each head has 48 features. remember that the query and key essentially contain a square matrix of all the token being attended to (in the context window), and value then decides within the context window, how to apply the attention and context window together.

$$
    H_r = \operatorname{softmax}\!\left(\frac{Q_rK_r^T}{\sqrt{d_h}} + C\right)V_r.
$$

softmax runs across key positions, which turns the values into probabilities (you could possibly interpret these as probabilities that the keys and queries are related or require attention on). the causal mask uses $C_{ij}=0$ for $j\leq i$ and $-\infty$ otherwise, so tokens cannot attend to the future. each head outputs $T \times d_h$; concatenating the heads gives $T \times d$, then a learned linear layer mixes their features.

the *feed-forward network* applies the same $768 \to 3072 \to 768$ transformation, with GELU between the two linear layers, independently to each token. both attention and feed-forward preserve the input shape, allowing the residual additions above. dropout is omitted here (the code defaults to `0.0`).


In [5]:
import torch
import torch.nn as nn

from typing import List

class DecoderBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, hidden_size: int, dropout=0.0):
        super().__init__()

        self.layer_norm_one = nn.LayerNorm(d_model)
        self.attention = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.layer_norm_two = nn.LayerNorm(d_model)
        # generally a wider layer than the embedding dimension is used here
        self.feed_forward = nn.Sequential(nn.Linear(d_model, hidden_size), nn.GELU(), nn.Linear(hidden_size, d_model), nn.Dropout(dropout))

    def forward(self, x: torch.Tensor):
        # x is (batch length, sequence length, embedding dimension)
        # sequence length (the number of tokens in the sequence) is what the transformer attends on
        T = x.size(1)

        # to remove the upper right triangle, which contains information about the future for row tokens
        causal_mask = torch.triu(torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1)

        # self-attention
        residual = x
        x = self.layer_norm_one(x)

        # causal attention, removing forward looking attention and does not return weights
        # masking makes future tokens -inf, which makes these probabilities 0 in the softmax operation
        attn_out, _ = self.attention(x, x, x, attn_mask=causal_mask, need_weights=False)
        x = residual + attn_out

        # feed forward
        residual = x
        x = self.feed_forward(self.layer_norm_two(x))
        x = residual + x

        return x

class GPTNeo(nn.Module):
    def __init__(self, n_layers: int, vocab_size: int, d_model: int, max_seq_len: int, hidden_size: List[int], n_heads: List[int], dropout: List[int]):
        super().__init__()

        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.pos_embedding = nn.Embedding(max_seq_len, d_model)
        assert n_layers == len(hidden_size) == len(n_heads) == len(dropout)
        self.blocks =  nn.ModuleList([DecoderBlock(d_model, n_heads[i], hidden_size[i], dropout[i]) for i in range(n_layers)])
        self.final_norm = nn.LayerNorm(d_model)
        self.logit = nn.Linear(d_model, vocab_size)

    def forward(self, x: torch.Tensor):
        x_seq_len = x.size(1)
        x_positions = torch.arange(x_seq_len, device=x.device).unsqueeze(0)

        embedding = self.token_embedding(x) + self.pos_embedding(x_positions)

        for block in self.blocks:
            embedding = block(embedding)

        logits = self.logit(self.final_norm(embedding))

        return logits
        

we can now pre-train the model to predict the next token. to fit within my RTX 5060 Ti's memory budget, this run uses **2 transformer blocks with 8 attention heads each**. both the embedding and feed-forward widths are 768, with a 512-token context and a 10,000-token vocabulary.

we train for **2 epochs** with batches of 32, using AdamW with a learning rate of `1e-3` and weight decay of `0.01`. mixed precision reduces memory use, and gradient clipping caps the gradient norm at 1.

the initial target is a cross-entropy loss below 2. validation loss will be needed to assess generalisation; comparisons with the paper also depend on tokenisation and evaluation setup.


In [6]:
from src.data.dataset import LMDataset
from torch.utils.data import DataLoader
from collections import deque

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tr_data = LMDataset('C:/data/tinystories/processed/train_gpt_neo.bin', 512)
tst_data = LMDataset('C:/data/tinystories/processed/test_gpt_neo.bin', 512)

tr_loader = DataLoader(tr_data, batch_size=32, shuffle=True, pin_memory=True, num_workers=0)
tst_loader = DataLoader(tst_data, batch_size=32, shuffle=True, pin_memory=True, num_workers=0)

In [10]:
epochs = 2
learning_rate = 1e-3

# model and training parameters
model = GPTNeo(2, 10_000, 768, 512, [768, 768], [8, 8], dropout=[0, 0]).to(device)
loss_fn = nn.CrossEntropyLoss()
optimiser = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)
scaler = torch.amp.GradScaler("cuda")

# training loop
loss_hist = []

model.train()
total_batches = len(tr_loader)
recent_losses = deque(maxlen=500)

for epoch in range(epochs):
    for batch_idx, (X_batch, y_batch) in enumerate(tr_loader, start=1):
        X_batch = X_batch.to(device, dtype=torch.long, non_blocking=True)
        y_batch = y_batch.to(device, dtype=torch.long, non_blocking=True)

        # flush gradients from previous iteration
        optimiser.zero_grad(set_to_none=True)

        # some floating point casting stuff ??
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            logits = model(X_batch)
            # -1 figures out the remaining dimensions before the vocab size layer
            loss = loss_fn(logits.reshape(-1, 10_000), y_batch.reshape(-1))

        # scale gradients since we have low precision floating points
        scaler.scale(loss).backward()
        scaler.unscale_(optimiser)
        # normalise gradient l2 norm to 1
        grad_norm = nn.utils.clip_grad_norm_(model.parameters(), max_norm=1)
        scaler.step(optimiser)
        scaler.update()

        recent_losses.append(loss.detach().item())
        recent_loss = sum(recent_losses) / len(recent_losses)

        loss_hist.append({ 'epoch': epoch, 'i': batch_idx, 'loss': loss.detach().item(), 'recent_loss': recent_loss, 'grad_norm': grad_norm.item() })

        if batch_idx % 1000 == 0 or batch_idx == total_batches:
            progress = 100 * batch_idx / total_batches

            print(
                f"batch {batch_idx}/{total_batches} ({progress:.1f}%) :: "
                f"recent loss: {recent_loss:.4f}",
                flush=True,
            )

torch.save(model.state_dict(), "C:/dev/storyteller/weights/20260925_768_768_8_8_0_0_transformer.pt")

batch 1000/28384 (3.5%) :: recent loss: 2.7874
batch 2000/28384 (7.0%) :: recent loss: 2.3199
batch 3000/28384 (10.6%) :: recent loss: 2.1454
batch 4000/28384 (14.1%) :: recent loss: 2.0554
batch 5000/28384 (17.6%) :: recent loss: 1.9923
batch 6000/28384 (21.1%) :: recent loss: 1.9420
batch 7000/28384 (24.7%) :: recent loss: 1.9027
batch 8000/28384 (28.2%) :: recent loss: 1.8816
batch 9000/28384 (31.7%) :: recent loss: 1.8525
batch 10000/28384 (35.2%) :: recent loss: 1.8322
batch 11000/28384 (38.8%) :: recent loss: 1.8097
batch 12000/28384 (42.3%) :: recent loss: 1.7966
batch 13000/28384 (45.8%) :: recent loss: 1.7865
batch 14000/28384 (49.3%) :: recent loss: 1.7718
batch 15000/28384 (52.8%) :: recent loss: 1.7606
batch 16000/28384 (56.4%) :: recent loss: 1.7441
batch 17000/28384 (59.9%) :: recent loss: 1.7371
batch 18000/28384 (63.4%) :: recent loss: 1.7280
batch 19000/28384 (66.9%) :: recent loss: 1.7208
batch 20000/28384 (70.5%) :: recent loss: 1.7147
batch 21000/28384 (74.0%) :: re

our training loss gets to 1.58, which is substantially lower than our best LSTM model. how does it fare in the test set?

In [5]:
# this is runnable without having ran the training step
model = GPTNeo(2, 10_000, 768, 512, [768, 768], [8, 8], dropout=[0, 0]).to(device)
loss_fn = nn.CrossEntropyLoss()
state_dict = torch.load('C:/dev/storyteller/weights/20260925_768_768_8_8_0_0_transformer.pt')
model.load_state_dict(state_dict)

total_loss = 0
total_tokens = 0

model.eval()

with torch.no_grad():
    for X_batch, y_batch in tst_loader:
        X_batch = X_batch.to(device, dtype=torch.long, non_blocking=True)
        y_batch = y_batch.to(device, dtype=torch.long, non_blocking=True)

        with torch.autocast(device_type='cuda', dtype=torch.float16):
            logits = model(X_batch)
            loss = loss_fn(logits.reshape(-1, 10_000), y_batch.reshape(-1))

        num_tokens = y_batch.numel()

        total_loss += loss.item() * num_tokens
        total_tokens += num_tokens

avg_loss = total_loss / total_tokens

print(f'average loss :: {avg_loss}')
print(f'perplexity :: {np.exp(avg_loss)}')

average loss :: 1.5897750075302977
perplexity :: 4.902645745852966


incredible stuff; we've reduced our perplexity from ~19 to ~5, which is a massive boost in prediction ability. instead of "randomly" (not exactly correct, but bear with me) choosing from 19 words, the model is on average choosing from 5 words. let's put it through some real story generation tasks, and compare how *natural* and *creative* the stories are compared to the LSTM models.

In [13]:
def generate_text(prompt: str, model: nn.Module, tokeniser: AutoTokenizer, device) -> str:
    model.eval()
    prompt_tokens = tokeniser.encode(prompt)
    prompt_tokens = torch.tensor(prompt_tokens, dtype=torch.long, device=device)

    build_prompt = prompt
    # get model prediction. adds a dimension of size 1 to the prompt tokens, which makes it [1, 4]
    # dummying the batch variable. then squeeze 0 removes that batch dimension
    while True:
        model_logits = model(prompt_tokens.unsqueeze(0)).squeeze(0)[-1]
        probs = torch.softmax(model_logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)

        # check if need to break
        if next_token.item() == tokeniser.eos_token_id: break

        next_word = tokeniser.decode(next_token.item())
        build_prompt += next_word

        prompt_tokens = tokeniser.encode(build_prompt)
        prompt_tokens = torch.tensor(prompt_tokens, dtype=torch.long, device=device)

    return build_prompt

import textwrap

def pretty_print(text: str, width: int = 80):
    clean_text = " ".join(text.split())
    print(textwrap.fill(clean_text, width=width))

the below is pretty impressive; the story not only perfectly captures the proper noun (Lily, a feminine name, and referring to the protagonist as 'she/her' throughout), but also carries forward the plot of the bananas to the end.

In [15]:
pretty_print(generate_text('Once upon a time,', model, tokeniser, device))

Once upon a time, there was a little girl named Lily. Lily was very happy and
loved Christmas. She dreamed of the big tree she had seen and the bright lights
in the sky. One day, she went with her mommy to get dressed. Her mommy took her
to the store and they bought yummy bananas. Lily was so excited to eat her
bananas that she started to sing her favorite song. Suddenly, her mommy
accidentally put a big slice of bananas on the table. It smelled so good that
she forgot about the fruit. Her mommy said "Are you sure you don't like that,
Lily?" Lily nodded her head and said "Yes mommy, I do." And from that day on
Lily always asked for the yummy bananas.


in comparison, the LSTM model produced the below response in initial research
```
Once upon a time, there was a little girl who loved to play outside. One day,
she saw that his blocks there was a thief. He was scared to run away and even
united. He picked up his package and ran to his mom.  The end the horse wouldn't
say possible skills, and the billboard treasured all right. He then asked his
friend, a little bird, surprise playing better it possible. The squirrel's bad
material.  The frog spent the game yummy they had than the tiny witch.
Everywhere he went, it had a shiny black paw on him. He was so happy. Before
something, it, the bee was just now small ornament and were talking about she
than she was to like her for your a warning.  But the King was it very special.
The shopkeeper had, so anyone was a few colors and bad friends. The boy saw dust
to their orange turkey from a fair lip, but all his friends were happy and
shaking, and then accepted the plant.  The turkey tried. He was toys to not look
so good.   The little girl explored it all the way home. She never liked to draw
the bug forever. She knew she would always try again the whole way home.
```
the first few sentences seem okay, but then the story starts talking about horses and frogs. let's try some more prompts.

In [17]:
pretty_print(generate_text('There once lived a girl named Gloria', model, tokeniser, device))

There once lived a girl named Gloria and her friends. One day Gloria found a
barrel in the woods. She was so excited! Gloria and her friends used the barrel
in her backpack. The barrel was full of treasures and it included them in her
special bow. Gloria was very excited and she ran home to show her mom and dad.
It was a weird surprise. But when her mom and dad did this the barrel jeller,
Gloria looked around. They laughed and said, "Gloria, these more treasures!"
Then they even recommended a game of hide and seek whenever she wanted. Gloria
was so happy and she couldn't wait to surprise her friends, have a weird barrel.


In [28]:
pretty_print(generate_text('A long, long time ago', model, tokeniser, device))

A long, long time ago, a little girl named Lily lived in a small house next
door. She loved to play outside, but she had not wear her collar. One day, Lily
found a black cat in her backyard. Lily was so surprised to see such a beautiful
cat that she decided to keep it on her own. Lily played with the cat for a
while, but she found more and more things in the house. She forgot about her
collar and went to play outside with her new collar. When she was in for a
while, it started to get dark outside. She was scared and did not know what to
do. with a quick thinking, Lily knew that her collar was gone. She ran inside to
find her friends. They found her mother and said, "Lily, you played with your
new little collar on the ground. You remembered your fun day in the garden.
Let's go back inside and get you some fixed!" Lily smiled and felt happy knowing
that her new home was safe from all the things in its life.


very cool stuff. this model vintage is saved `weights/20260925_768_768_8_8_0_0_transformer.pt`. let's do a quick calculation for how many weights our model (ex-embedding) has.
- embedding dimension of $768$
- 8 head transformer, thus splits the attention weight matrices $768 / 8 = 96 \times 768$. 8 heads of these attention heads is $589,824$ parameters per block
- the feedforward network is $768 \to 768 \to 768$, and thus adds $768 \times 768 + 768 \times 768 = 1,179,648$

and thus each transformer block contains ~1.8m parameters. with two blocks, this is ~3.6m parameters excluding the output logit layer. we can go deeper (and wider!). we make the following changes:
1. make the hidden size of the linear layer wider to 3024 neurons. this is the original implementation of the paper.
2. add two more transformer blocks
3. lower the batch size to 16 to allow for appropiate compute
4. change the floating point type to `bfloat16`, which increases the numeric range but lowers precision (removes the need for a scaler)

In [7]:
tr_loader = DataLoader(tr_data, batch_size=32, shuffle=True, pin_memory=True, num_workers=0)
tst_loader = DataLoader(tst_data, batch_size=32, shuffle=True, pin_memory=True, num_workers=0)

In [8]:
epochs = 3
learning_rate = 1e-3

# model and training parameters
model = GPTNeo(4, 10_000, 768, 512, [3024] * 4, [8] * 4, dropout=[0] * 4).to(device)
loss_fn = nn.CrossEntropyLoss()
optimiser = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01, fused=True)

# training loop
loss_hist = []

model.train()
total_batches = len(tr_loader)
recent_losses = deque(maxlen=500)

for epoch in range(epochs):
    for batch_idx, (X_batch, y_batch) in enumerate(tr_loader, start=1):
        X_batch = X_batch.to(device, dtype=torch.long, non_blocking=True)
        y_batch = y_batch.to(device, dtype=torch.long, non_blocking=True)

        # flush gradients from previous iteration
        optimiser.zero_grad(set_to_none=True)

        # some floating point casting stuff ??
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            logits = model(X_batch)
            # -1 figures out the remaining dimensions before the vocab size layer
            loss = loss_fn(logits.reshape(-1, 10_000), y_batch.reshape(-1))

        loss.backward()
        # normalise gradient l2 norm to 1
        grad_norm = nn.utils.clip_grad_norm_(model.parameters(), max_norm=1)
        optimiser.step()
    
        recent_losses.append(loss.detach().item())
        recent_loss = sum(recent_losses) / len(recent_losses)

        loss_hist.append({ 'epoch': epoch, 'i': batch_idx, 'loss': loss.detach().item(), 'recent_loss': recent_loss, 'grad_norm': grad_norm.item() })

        if batch_idx % 1000 == 0 or batch_idx == total_batches:
            progress = 100 * batch_idx / total_batches

            print(
                f"batch {batch_idx}/{total_batches} ({progress:.1f}%) :: "
                f"recent loss: {recent_loss:.4f}",
                flush=True,
            )

torch.save(model.state_dict(), "C:/dev/storyteller/weights/20260926_3024_4x_8_4x_0_4x_transformer.pt")

batch 1000/28384 (3.5%) :: recent loss: 2.4583
batch 2000/28384 (7.0%) :: recent loss: 2.0380
batch 3000/28384 (10.6%) :: recent loss: 1.8918
batch 4000/28384 (14.1%) :: recent loss: 1.8026
batch 5000/28384 (17.6%) :: recent loss: 1.7505
batch 6000/28384 (21.1%) :: recent loss: 1.7055
batch 7000/28384 (24.7%) :: recent loss: 1.6743
batch 8000/28384 (28.2%) :: recent loss: 1.6511
batch 9000/28384 (31.7%) :: recent loss: 1.6268
batch 10000/28384 (35.2%) :: recent loss: 1.6083
batch 11000/28384 (38.8%) :: recent loss: 1.5906
batch 12000/28384 (42.3%) :: recent loss: 1.5735
batch 13000/28384 (45.8%) :: recent loss: 1.5636
batch 14000/28384 (49.3%) :: recent loss: 1.5516
batch 15000/28384 (52.8%) :: recent loss: 1.5364
batch 16000/28384 (56.4%) :: recent loss: 1.5277
batch 17000/28384 (59.9%) :: recent loss: 1.5220
batch 18000/28384 (63.4%) :: recent loss: 1.5098
batch 19000/28384 (66.9%) :: recent loss: 1.5032
batch 20000/28384 (70.5%) :: recent loss: 1.4965
batch 21000/28384 (74.0%) :: re

In [11]:
# this is runnable without having ran the training step
model = GPTNeo(4, 10_000, 768, 512, [3024] * 4, [8] * 4, dropout=[0] * 4).to(device)
loss_fn = nn.CrossEntropyLoss()
state_dict = torch.load('C:/dev/storyteller/weights/20260926_3024_4x_8_4x_0_4x_transformer.pt')
model.load_state_dict(state_dict)

total_loss = 0
total_tokens = 0

model.eval()

with torch.no_grad():
    for X_batch, y_batch in tst_loader:
        X_batch = X_batch.to(device, dtype=torch.long, non_blocking=True)
        y_batch = y_batch.to(device, dtype=torch.long, non_blocking=True)

        with torch.autocast(device_type='cuda', dtype=torch.float16):
            logits = model(X_batch)
            loss = loss_fn(logits.reshape(-1, 10_000), y_batch.reshape(-1))

        num_tokens = y_batch.numel()

        total_loss += loss.item() * num_tokens
        total_tokens += num_tokens

avg_loss = total_loss / total_tokens

print(f'average loss :: {avg_loss}')
print(f'perplexity :: {np.exp(avg_loss)}')

average loss :: 1.356037156920611
perplexity :: 3.88078385191513


the deeper, wider network with approx ~6.4m parammeters materially improves the average loss and perplexity by `~0.24` and `~1.1` respectively. this gets us very close to the loss found in the paper. it is likely that they spent additional effort in the optimisation of training parameters; with my compute, this optimisation is possible, but is quite tedious. i am fairly satisfied with these results. how does the updated model fair in story generation?

In [21]:
pretty_print(generate_text('Once upon a time', model, tokeniser, device))

Once upon a time, there was a boy named Timmy who loved to watch the season. He
loved to look at the big snowflakes and the colorful leaves. One day, Timmy
decided to spend the night outside watching the snow fall from the window. He
looked up at the sky and thought about how cozy it was to watch the snowflakes
fall. He loved to watch the colorful snowflakes move around in the cold air.
Timmy felt happy and not scared anymore.


In [22]:
pretty_print(generate_text('A long, long time ago', model, tokeniser, device))

A long, long time ago, there was a magical land filled with any surprises. With
every surprise the world had to offer, the special things that these toys saw
would fill it with joy and their land became brighter. After hours of walking,
the cobs began to grow as the best of what the land had to offer and energy
their distant land was worth for. This story is to be kind, understanding and
understanding. Everyone has the ability to remember their plan and the distant
land".
